# چالش: تحلیل متن درباره علم داده

در این مثال، بیایید یک تمرین ساده انجام دهیم که تمام مراحل یک فرایند سنتی علم داده را پوشش می‌دهد. نیازی به نوشتن کد ندارید، فقط می‌توانید روی سلول‌های زیر کلیک کنید تا آنها را اجرا کرده و نتیجه را مشاهده کنید. به‌عنوان یک چالش، شما تشویق می‌شوید که این کد را با داده‌های مختلف امتحان کنید.

## هدف

در این درس، ما درباره مفاهیم مختلف مرتبط با علم داده صحبت کرده‌ایم. بیایید با انجام یک عملیات **متن‌کاوی** سعی کنیم مفاهیم مرتبط بیشتری را کشف کنیم. ابتدا متنی درباره علم داده خواهیم داشت، سپس کلیدواژه‌ها را از آن استخراج می‌کنیم و در پایان سعی می‌کنیم نتیجه را به تصویر بکشیم.

به‌عنوان متن، از صفحه علم داده در ویکی‌پدیا استفاده خواهم کرد:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## مرحله ۱: دریافت داده‌ها

اولین مرحله در هر فرایند علم داده‌ها، دریافت داده‌ها است. ما برای این کار از کتابخانه `requests` استفاده خواهیم کرد:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## مرحله ۲: تبدیل داده‌ها

گام بعدی تبدیل داده‌ها به شکلی است که برای پردازش مناسب باشد. در مورد ما، کد منبع HTML صفحه را دانلود کرده‌ایم و باید آن را به متن ساده تبدیل کنیم.

روش‌های زیادی برای انجام این کار وجود دارد. ما از [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/)، یک کتابخانه محبوب پایتون برای تجزیه HTML استفاده خواهیم کرد. BeautifulSoup به ما اجازه می‌دهد عناصر خاصی از HTML را هدف بگیریم، بنابراین می‌توانیم روی محتوای اصلی مقاله ویکی‌پدیا تمرکز کنیم و برخی منوهای ناوبری، نوارهای کناری، پاورقی‌ها و سایر محتوای نامرتبط را کاهش دهیم (اگرچه ممکن است مقداری متن قالب هنوز باقی بماند).


ابتدا، باید کتابخانه BeautifulSoup را برای تجزیه HTML نصب کنیم:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## مرحله ۳: دریافت بینش‌ها

مهم‌ترین مرحله تبدیل داده‌های ما به شکلی است که بتوانیم از آن‌ها بینش استخراج کنیم. در مورد ما، می‌خواهیم کلیدواژه‌ها را از متن استخراج کنیم و ببینیم کدام کلیدواژه‌ها معنی‌دارتر هستند.

ما از کتابخانه پایتون به نام [RAKE](https://github.com/aneesha/RAKE) برای استخراج کلیدواژه استفاده خواهیم کرد. ابتدا، بیایید این کتابخانه را در صورت نبود نصب کنیم:


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

عملکرد اصلی از شیء `Rake` در دسترس است، که می‌توانیم آن را با استفاده از برخی پارامترها سفارشی کنیم. در مورد ما، حداقل طول یک کلمه کلیدی را ۵ کاراکتر، حداقل فراوانی کلمه کلیدی در سند را ۳ و حداکثر تعداد کلمات در یک کلمه کلیدی را ۲ تنظیم می‌کنیم. می‌توانید با مقادیر دیگر آزمایش کنید و نتیجه را مشاهده کنید.


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


ما فهرستی از واژه‌ها به همراه درجه اهمیت مرتبط با آن‌ها به‌دست آوردیم. همان‌طور که مشاهده می‌کنید، رشته‌های مرتبط‌تر، مانند یادگیری ماشین و داده‌های بزرگ، در جایگاه‌های بالای فهرست حضور دارند.

## گام ۴: مصورسازی نتیجه

مردم بهترین تفهیم را زمانی از داده‌ها دارند که به صورت بصری باشند. بنابراین اغلب منطقی است که داده‌ها را برای استخراج برخی بینش‌ها مصورسازی کنیم. می‌توانیم از کتابخانه `matplotlib` در پایتون برای ترسیم توزیع ساده کلیدواژه‌ها با اهمیت آن‌ها استفاده کنیم:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

با این حال، یک روش بهتر برای تجسم فرکانس کلمات وجود دارد - استفاده از **ابر کلمات**. ما باید یک کتابخانه دیگر را برای رسم ابر کلمات از فهرست کلیدواژه‌های خود نصب کنیم.


In [ ]:
!{sys.executable} -m pip install wordcloud

شیء `WordCloud` مسئول دریافت متن اصلی یا لیست از پیش محاسبه شده‌ای از کلمات با فراوانی‌هایشان است و تصویری بازمی‌گرداند که سپس می‌توان با استفاده از `matplotlib` آن را نمایش داد:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

ما همچنین می‌توانیم متن اصلی را به `WordCloud` بدهیم — بیایید ببینیم آیا می‌توانیم نتیجه مشابهی بگیریم:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

اکنون می‌توانید ببینید که ابر کلمات چگونه چشمگیرتر به نظر می‌رسد، اما همچنین شامل نویز زیادی است (مثلاً کلمات نامرتبط مانند `Retrieved on`). همچنین، کلیدواژه‌های کمتری متشکل از دو کلمه داریم، مانند *data scientist* یا *computer science*. این به این دلیل است که الگوریتم RAKE در انتخاب کلیدواژه‌های خوب از متن عملکرد بسیار بهتری دارد. این مثال اهمیت پیش‌پردازش و پاک‌سازی داده‌ها را نشان می‌دهد، زیرا تصویر واضح در انتها به ما اجازه می‌دهد تصمیمات بهتری بگیریم.

در این تمرین ما یک فرآیند ساده استخراج معنی از متن ویکی‌پدیا را به صورت کلیدواژه‌ها و ابر کلمات انجام دادیم. این مثال بسیار ساده است، اما به خوبی تمام مراحل معمولی که یک دانشمند داده هنگام کار با داده‌ها انجام می‌دهد را نشان می‌دهد، از جمع‌آوری داده‌ها تا تصویرسازی.

در دوره ما تمام این مراحل را به تفصیل بررسی خواهیم کرد.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
